# Uji nyata Sorong + T7SL — Gate 2

**Persiapan notebook, bukan hasil run lokasi nyata.** Publikasi otomatis melewati
setup dan engine. Untuk memakai notebook: pilih **GPU** di Settings, aktifkan
Secret **LocatioIntelligent** pada Add-ons, ubah `ENABLE_NOTEBOOK_SETUP=True`,
lalu jalankan cell di bawah. Internet sudah diaktifkan pada metadata notebook.

Latitude **-0.8994976613947719**, longitude **131.3221639428227**. Dataset
`chrisbiran/t7sl-traffic-videos` sudah attached. Tidak ada input folder/model/runtime.

Setelah isi kuis: **Simpan jawaban → Periksa kesiapan → Mulai uji**.
Runtime/model otomatis disiapkan saat Mulai uji. Area dan Traffic independen.
Traffic memakai kandidat FP32: YOLO batch4, ReID IO binding/window8/crop32.
Satu GPU memproses satu video; dua GPU memproses dua video bersamaan.
Total counting uji sama, tetapi identitas track belum setara baseline.
Sumber Area berasal dari PBF nasional GCS dan BQ; tidak perlu registry baru.
Tahap ini dapat mengunduh PBF nasional ~1,74GB, mengekstrak sumber Sorong dan
mengambil snapshot BQ berbudget. Internet/izin IAM/BQ harus tersedia.

Policy MVP: alokasi bangunan calon x4; Residential40/60 atau35/45/20; Fasum jarak;
sepeda tidak digunakan; video300–306detik diterima tanpa skala; hasil PARTIAL
boleh dipakai dan tetap berlabel. Lookup Residential/AE18 ditiadakan, faktor1.
**Sesuai keputusan owner, jam>15 tetap memerlukan sesi tambahan.** Anda tetap
bisa menguji Area/Traffic; PPAC akan BLOCKED untuk18jam dengan delapan video ini.
Untuk uji gabungan delapan video sampai PPAC, gunakan jam operasional <=15.


In [ ]:
# Satu sakelar untuk mengaktifkan notebook di editor Anda.
# Publikasi Kaggle tidak mengeksekusi setup/routing/inference.
ENABLE_NOTEBOOK_SETUP=False
print('Notebook siap dibuka; engine hanya berjalan setelah klik Mulai uji.')


In [ ]:
if ENABLE_NOTEBOOK_SETUP:
    import base64,hashlib,zipfile,io,json,sys,subprocess,os,importlib,importlib.metadata
    from pathlib import Path
    PAYLOAD=base64.b64decode('...')
    assert hashlib.sha256(PAYLOAD).hexdigest()=='5afea0a951c5ba8a61a726d106c842b2df6258827349d77b61e087e0f0938237'
    BASE=Path('/kaggle/working/gate2');BASE.mkdir(parents=True,exist_ok=True)
    archive=zipfile.ZipFile(io.BytesIO(PAYLOAD))
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in archive.namelist())
    archive.extractall(BASE)
    # Exact pinned path; old0.4.0/0.4.1files may exist, but are never selected.
    WHEEL=BASE/'package'/'traffic_location_engines-0.5.3rc1-py3-none-any.whl'
    assert hashlib.sha256(WHEEL.read_bytes()).hexdigest()=='2b7ac3c7b9250e2e7c762245542462d1c42563f879e4645ad58adeda630f6aeb'
    subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(WHEEL)],check=True)
    # Re-running a notebook must not retain the prior imported engine modules.
    for name in list(sys.modules):
        if name=='traffic_location' or name.startswith('traffic_location.'):del sys.modules[name]
    importlib.invalidate_caches()
    assert importlib.metadata.version('traffic-location-engines')=='0.5.3rc1'
    import traffic_location
    with zipfile.ZipFile(WHEEL) as package:
        package_root=Path(traffic_location.__file__).parent
        for name in package.namelist():
            if name.startswith('traffic_location/') and name.endswith('.py'):
                installed=package_root/Path(name).relative_to('traffic_location')
                assert installed.read_bytes()==package.read(name),'Installed source differs: '+name
    subprocess.run([sys.executable,'-m','pip','install','google-cloud-storage==3.6.0','ipywidgets>=8,<9'],check=True)
    from traffic_location.artifacts import read,write
    from traffic_location.integration.kaggle_runtime import KaggleRuntime
    PERFORMANCE_EXPERIMENT={'reuse':True}  # candidate FP32; one video per visible GPU
    CONFIG=BASE/'config';LATITUDE=-0.8994976613947719;LONGITUDE=131.3221639428227
    print('Wheel0.5.3rc1 dan seluruh source terverifikasi. Belum ada engine berjalan.')
else:
    print("PREPARED ONLY: cell dilewati.")


## Kuis bisnis
Pilih profil jalan PPAC secara eksplisit; jenis OSM hanya referensi. Config sumber,
GPU, GDAL, model dan folder ditangani sistem. Nilai faktor1 adalah pilihan user
bila memang tidak ada penyesuaian. J96/AE18 tidak lagi ditanyakan.


In [ ]:
if ENABLE_NOTEBOOK_SETUP:
    import ipywidgets as W
    from IPython.display import display,FileLink
    from datetime import datetime
    from zoneinfo import ZoneInfo
    mode=W.Dropdown(description='Uji',options=[('Pilih mode',None),('Area saja','area'),('Traffic saja','traffic'),('Area + Traffic + PPAC','workflow')])
    target=W.Text(description='Kode toko',value='T7SL')
    osm=W.Dropdown(description='Jalan OSM',options=[('Pilih (referensi)',None)]+[(s,s) for s in ['motorway','trunk','primary','secondary','tertiary','unclassified','residential','living_street','service']])
    road=W.Dropdown(description='Profil PPAC',options=[('Pilih',None)]+[(s,s) for s in ['highway','mainstreet','street','substreet']])
    days=W.Dropdown(description='Hari/minggu',options=[('Pilih',None)]+[(str(i),i) for i in range(1,8)])
    hours=W.Dropdown(description='Jam/hari',options=[('Pilih',None)]+[(str(i),i) for i in range(1,25)])
    drive=W.Dropdown(description='Drive thru',options=[('Pilih',None),('Tidak',False),('Ya',True)])
    def factor(label):
        return W.Dropdown(description=label,options=[('Pilih',None),('1 — tanpa penyesuaian',1.),('Isi angka lain','manual')]),W.FloatText(value=1.,disabled=True)
    swot,swot_n=factor('SWOT');cut,cut_n=factor('Potongan');park,park_n=factor('Parkir')
    for choice,number in [(swot,swot_n),(cut,cut_n),(park,park_n)]:
        choice.observe(lambda change,number=number:setattr(number,'disabled',change['new']!='manual'),names='value')
    confirm=W.Checkbox(value=False,indent=False,description='Saya menyatakan video T7SL dipakai untuk analisis target Sorong di atas.')
    save=W.Button(description='Simpan jawaban',button_style='primary')
    check=W.Button(description='Periksa kesiapan',button_style='info')
    start=W.Button(description='Mulai uji',button_style='success')
    output=W.Output();ANSWERS=None;RUNTIME=None;LAST_RESULT=None
    
    def save_answers(_):
        global ANSWERS,RUNTIME
        value=lambda c,n:n.value if c.value=='manual' else c.value
        params={'schema_version':'ppac_parameters_mvp_v2',
          'residential':{'operating_days_per_week':days.value,'swot_factor':value(swot,swot_n)},
          'traffic':{'road_type':road.value,'operating_hours':hours.value,'drive_thru':drive.value},
          'aggregation':{'traffic_cut_factor':value(cut,cut_n),'parking_factor':value(park,park_n)},
          'provenance':'Owner MVP; business answers explicitly saved; Residential/AE18 lookup disabled'}
        ANSWERS={'mode':mode.value,'target_id':target.value.strip(),'road_osm':osm.value,'parameters':params,
          'target_confirmed':confirm.value,'analysis_date':datetime.now(ZoneInfo('Asia/Bangkok')).date().isoformat()}
        assert ANSWERS['target_id'],'Kode toko wajib diisi.'
        RUNTIME=KaggleRuntime(BASE/'system',CONFIG,LATITUDE,LONGITUDE,target_id=ANSWERS['target_id'],period_id=ANSWERS['analysis_date'],performance=PERFORMANCE_EXPERIMENT)
        write(BASE/'answers.json',ANSWERS)
        with output:
            output.clear_output();print('Jawaban tersimpan. Belum ada engine berjalan.')
            if hours.value and hours.value>15:print('PPAC >15jam menunggu sesi tambahan; Area/Traffic tetap dapat diuji.')
    
    
    def show_report(_):
        with output:
            try:
                assert RUNTIME is not None,'Simpan jawaban terlebih dahulu.'
                print('Memeriksa GCS sekarang (metadata saja; belum download PBF/model)...')
                cloud=RUNTIME.probe();inventory=RUNTIME.video_inventory()
                display(W.HTML('<b>Status GCS: '+cloud['gcs']['status']+'</b>'))
                for name,item in cloud['gcs']['objects'].items():print(name,':',item['status'],item.get('code',''))
                print('Sumber Area:',cloud['gcs']['area_source'],'— bundle Sorong akan dibangun saat Mulai uji.')
                for item in inventory:print(item['id'],item['status'],'durasi',round(item['duration_seconds'],3),'detik')
                print('Runtime/model belum dihitung READY sebelum setup. Akan disiapkan otomatis saat Mulai uji.')
                if ANSWERS['parameters']['traffic']['operating_hours'] and ANSWERS['parameters']['traffic']['operating_hours']>15:
                    print('PPAC akan BLOCKED untuk jam>15 tanpa sesi tambahan, sesuai keputusan Anda.')
            except Exception as exc:print('Pemeriksaan belum berhasil:',getattr(exc,'code',type(exc).__name__),str(exc) if isinstance(exc,AssertionError) else 'Periksa Add-ons Secret / Internet / izin GCS. Bukan dianggap data kosong.')
    
    
    def start_test(_):
        global LAST_RESULT
        with output:
            start.disabled=True
            try:
                assert RUNTIME is not None and ANSWERS['mode'] is not None,'Simpan jawaban dan pilih mode.'
                if ANSWERS['mode']=='workflow':
                    assert ANSWERS['target_confirmed'],'Konfirmasi target Sorong untuk penggabungan.'
                    # Do not waste engine time on missing business answers; >15 remains
                    # allowed for engine testing and PPAC returns its specific block later.
                    def missing(v):
                        if isinstance(v,dict):return any(missing(x) for x in v.values())
                        return v is None
                    assert not missing(ANSWERS['parameters']),'Lengkapi input bisnis PPAC di kuis.'
                print('Pemeriksaan GCS aktual sebelum engine...');RUNTIME.probe()
                LAST_RESULT=RUNTIME.execute(ANSWERS['mode'],ANSWERS['parameters'],target_confirmed=ANSWERS['target_confirmed'])
                print('HASIL TIAP ENGINE/VIDEO:')
                for row in LAST_RESULT['engines']:
                    print(row['id'],row['execution_status'],row.get('quality',''),row.get('code',''))
                    if 'summary' in row:print(json.dumps(row['summary'],ensure_ascii=False))
                    elif 'message' in row:print(row['message'])
                print('PPAC:',LAST_RESULT['ppac']['execution_status'],'STD:',LAST_RESULT['ppac'].get('std_total'))
                for diagnostic in LAST_RESULT['ppac'].get('diagnostics',[]):print(diagnostic)
                path=Path(LAST_RESULT['output']);print('Folder hasil/log:',path)
                for file in [path/'summary.json',path/'progress.json',BASE/'system/system-status.json',BASE/'system/area-preparation.log']:
                    if file.exists():display(FileLink(str(file)))
            except Exception as exc:
                print('Uji belum dimulai/terhenti:',getattr(exc,'code',type(exc).__name__),str(exc) if isinstance(exc,AssertionError) else 'Lihat output setup dan system-status.json.')
            finally:start.disabled=False
    save.on_click(save_answers);check.on_click(show_report);start.on_click(start_test)
    display(W.VBox([W.HTML(f'<b>Sorong: {LATITUDE}, {LONGITUDE}</b>'),mode,target,osm,road,days,hours,drive,
     W.HBox([swot,swot_n]),W.HBox([cut,cut_n]),W.HBox([park,park_n]),confirm,W.HBox([save,check,start]),output]))
else:
    print("PREPARED ONLY: cell dilewati.")


## Arti status dan hasil
- **NOT_CHECKED:** belum dilakukan permintaan ke GCS.
- **NOT_FOUND:** API GCS mengembalikan object tidak ditemukan; tidak disamakan dengan izin gagal.
- **ACCESS_DENIED / ERROR:** akses atau pemeriksaan gagal, bukan sumber kosong.
- **READY:** metadata sumber ditemukan; SHA penuh tetap diverifikasi saat staging.
- **BUILDABLE_FROM_PINNED_PBF:** PBF tersedia; sumber Sorong belum dihitung,
  akan disiapkan dari PBF dan BQ, bukan diminta kepada user.

Runtime dipasang otomatis pada percobaan engine terkait. Model GCS yang belum
tersedia di-bootstrap dari rilis resmi yang SHA-nya terkunci; write-cache GCS
gagal tidak menghilangkan input lokal terverifikasi. Error izin membaca GCS
bukan alasan mengganti model. Tidak ada data sintetis pada jalur ini.

Area dan tiap video diisolasi: kegagalan satu tahap tidak membatalkan hasil yang
sukses. PPAC hanya menggabungkan seluruh hasil wajib yang ada; jika ada engine
atau video gagal, STD tetap null dan hasil lain tetap terlihat.

**Batas uji:** GCS, BigQuery, GPU, coverage sumber Sorong dan akses model baru
terbukti setelah tombol dijalankan sendiri pada akun Kaggle Anda. Persiapan
notebook dan tests lokal tidak dianggap bukti bahwa data Sorong sudah dihitung.


## Eksperimen performa satu GPU (opsional)

Batch1 FP32 tetap baseline. Reuse hanya untuk stage selesai yang checksum-nya
valid dalam `/kaggle/working/gate2/system/completed-cache`; cache lokal tidak
bertahan bila session/disk Kaggle dihapus. File rusak ditolak, bukan dipakai.
Progress FPS/ETA diteruskan dari worker; timing counting ada di artefak cache.

Cell berikut menguji prefix256 frame, batch1 versus batch4, masing-masing tiga
fresh run berurutan pada cuda:0. Tidak mengubah profil run utama dan tidak
meluluskan otomatis. Jalankan setelah simpan jawaban. Ini uji teknis prefix,
bukan pengukuran akurasi atau bukti full-video. Tinjau report sebelum memilih
batch4 untuk uji lebih panjang. Semua setup/engine tetap OFF saat publikasi.


In [ ]:
if ENABLE_NOTEBOOK_SETUP:
    RUN_PERFORMANCE_TRIAL=False
    if RUN_PERFORMANCE_TRIAL:
        assert RUNTIME is not None,'Simpan jawaban terlebih dahulu.'
        from traffic_location.traffic.experiments.benchmark import trial
        from copy import deepcopy
        RUNTIME.connect();RUNTIME.setup_runtime('traffic')
        model_paths=RUNTIME.prepare_models()
        item=next(v for v in RUNTIME.video_inventory() if v['status']=='READY')
        profile=deepcopy(read(CONFIG/'fresh_request.json'))
        profile['input_root']=str(BASE)
        profile['artifacts']['video']={'path':item['path'],'sha256':item['slot']['sha256']}
        for name in ('detector','reid'):profile['artifacts'][name]['path']=model_paths[name]
        trial_root=BASE/'performance-trials'/datetime.now().strftime('%Y%m%dT%H%M%S_%f')
        trial_root.parent.mkdir(parents=True,exist_ok=True)
        producer=trial_root.with_suffix('.request.json');write(producer,profile)
        report=trial(producer,trial_root,geometry=CONFIG/'geometry'/(item['id']+'.json'),
                     policy=CONFIG/'policy.json',frames=256,batch=4,repetitions=3,device='cuda:0')
        print(json.dumps(report,indent=2));display(FileLink(str(trial_root/'report.json')))
    else:
        print('Eksperimen performa OFF; tidak ada fresh inference.')
else:
    print("PREPARED ONLY: cell dilewati.")
